In [1]:
import sys
import os
sys.path.append(os.path.abspath('../../'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from functions.preproc import *
from functions.ensembles import *

from sklearn import set_config
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import make_scorer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedKFold
from sksurv.metrics import integrated_brier_score
from sksurv.metrics import concordance_index_ipcw
from sksurv.nonparametric import kaplan_meier_estimator
from sklearn.inspection import permutation_importance

from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

import time
set_config(display="text")  # displays text representation of estimators

In [2]:
dataset = pd.read_csv("../data/data_final.csv")

diagnosis_variables = []

with open("../data/diagnosis_final.txt", "r") as file:
    for line in file:
        diagnosis_variables.append(line.strip())

treatment_variables = []
with open("../data/treatment_final.txt", "r") as file:
    for line in file:
        treatment_variables.append(line.strip())

survival_dataset = dataset.drop('Unnamed: 0', axis=1)

feature_columns = treatment_variables + diagnosis_variables

In [3]:
# Find duplicated patients
mask = (survival_dataset.duplicated(subset='PERSON_ID', keep=False)) #& (survival_dataset['Regional LN recurrence'] == 1)

# For each duplicated PERSON_ID with recurrence, keep only the row with the lowest survival days
to_remove = (
    survival_dataset[mask]
    .sort_values(['PERSON_ID', 'Survival days'], ascending=[True, True])
    .groupby('PERSON_ID')
    .head(-1)
    .index
)

survival_dataset = survival_dataset.drop(to_remove)

In [4]:
# Prepare data
data_x = survival_dataset[feature_columns]
data_y = Surv.from_arrays(
    event=survival_dataset['Regional LN recurrence'],
    time=survival_dataset['Survival days']
)

In [5]:
sgkf = StratifiedKFold(n_splits=5, shuffle=True, random_state=12)
cv = sgkf.split(X=data_x, y=survival_dataset['Regional LN recurrence'])


train_index, test_index = next(cv)
x_train = data_x.iloc[train_index]
y_train = data_y[train_index]

x_test = data_x.iloc[test_index]
y_test = data_y[test_index]

subsampling = survival_dataset.iloc[train_index].copy()

In [17]:
for train_idx, val_idx in cv:
    print("Train events:", survival_dataset.iloc[train_idx]["Regional LN recurrence"].sum())
    print("Val events:", survival_dataset.iloc[val_idx]["Regional LN recurrence"].sum())

Train events: 27.0
Val events: 6.0
Train events: 26.0
Val events: 7.0
Train events: 26.0
Val events: 7.0
Train events: 26.0
Val events: 7.0


In [ ]:
param_grid = {
    'learning_rate': [0.05, 0.1, 0.2],
    'n_estimators': [100, 200, 500],
    'max_depth': [1, 2, 4],
    'min_samples_split': [2, 5, 10],
}

sgkf_1 = StratifiedKFold(n_splits=4, shuffle=True, random_state=12)
cv = sgkf_1.split(X=x_train, y=y_train['event'])

In [19]:
results_df, best_params, summary = generic_grid_search(
    GradientBoostingSurvivalAnalysis,
    x_train,
    y_train,
    cv,
    param_grid,
    metrics=['c-index', 'c-index-ipcw', 'int-brier-score'],
    model_name='GBM',
    scale_data=False,
    scaler_type='standard',
    verbose=True,
    random_state=12,
    n_jobs=-1
)


GBM GRID SEARCH (Manual Implementation)
Model: GradientBoostingSurvivalAnalysis
Scaling: No
Metrics: ['c-index', 'c-index-ipcw', 'int-brier-score']
Parameter Grid:
  - learning_rate: [0.05, 0.1, 0.2]
  - n_estimators: [100, 200, 500]
  - max_depth: [1, 2, 4]
  - min_samples_split: [2, 5, 10]
Total combinations: 81

[1/81] Testing params: {'learning_rate': 0.05, 'n_estimators': 100, 'max_depth': 1, 'min_samples_split': 2}
  c-index: 0.7704 ± 0.0464
  c-index-ipcw: 0.5771 ± 0.0924
  int-brier-score: 0.0093 ± 0.0021
[2/81] Testing params: {'learning_rate': 0.05, 'n_estimators': 100, 'max_depth': 1, 'min_samples_split': 5}
  Elapsed: 0.2min, Est. remaining: 19.4min
  c-index: 0.7704 ± 0.0464
  c-index-ipcw: 0.5771 ± 0.0924
  int-brier-score: 0.0093 ± 0.0021
[3/81] Testing params: {'learning_rate': 0.05, 'n_estimators': 100, 'max_depth': 1, 'min_samples_split': 10}
  Elapsed: 0.5min, Est. remaining: 20.6min
  c-index: 0.7704 ± 0.0464
  c-index-ipcw: 0.5771 ± 0.0924
  int-brier-score: 0.009

In [24]:
# View the results DataFrame
results_df.sort_values(by='c-index_mean', ascending=False)

,learning_rate,n_estimators,max_depth,min_samples_split,c-index_mean,c-index_std,c-index-ipcw_mean,c-index-ipcw_std,int-brier-score_mean,int-brier-score_std
77,0.20,500,2,10,0.848635,0.066163,0.735927,0.164908,0.009167,0.002022
76,0.20,500,2,5,0.848635,0.066163,0.735927,0.164908,0.009167,0.002022
75,0.20,500,2,2,0.848635,0.066163,0.735927,0.164908,0.009167,0.002022
50,0.10,500,2,10,0.834182,0.039665,0.706028,0.143469,0.009235,0.002050
49,0.10,500,2,5,0.834182,0.039665,0.706028,0.143469,0.009235,0.002050
...,...,...,...,...,...,...,...,...,...,...
34,0.10,100,4,5,0.659277,0.199164,0.563491,0.088804,0.009754,0.001338
17,0.05,200,4,10,0.649576,0.192248,0.593111,0.066429,0.009854,0.001317
15,0.05,200,4,2,0.648608,0.194488,0.560782,0.076440,0.009717,0.001329
33,0.10,100,4,2,0.647478,0.194116,0.554422,0.082176,0.009733,0.001316


In [25]:
results_df.sort_values(by='c-index_mean', ascending=False).to_csv("../results/gbm_grid_search_results.csv", index=False)

# Train and Test Final Model with Best Hyperparameters

In [6]:
# Load best hyperparameters from grid search results
best_results = pd.read_csv("../results/normal/gbm_grid_search_results.csv")
# Select row with highest c-index-ipcw_mean
best_params = best_results.sort_values('c-index-ipcw_mean', ascending=False).iloc[0]

print("Best hyperparameters:")
print(f"  learning_rate: {best_params['learning_rate']}")
print(f"  n_estimators: {int(best_params['n_estimators'])}")
print(f"  max_depth: {int(best_params['max_depth'])}")
print(f"  min_samples_split: {int(best_params['min_samples_split'])}")
print(f"  c-index-ipcw_mean (CV): {best_params['c-index-ipcw_mean']:.4f}")
print()

Best hyperparameters:
  learning_rate: 0.2
  n_estimators: 500
  max_depth: 2
  min_samples_split: 10
  c-index-ipcw_mean (CV): 0.7359



In [7]:
# Train final model with best hyperparameters (NO SUBSAMPLING)
from sksurv.ensemble import GradientBoostingSurvivalAnalysis

final_model = GradientBoostingSurvivalAnalysis(
    learning_rate=best_params['learning_rate'],
    n_estimators=int(best_params['n_estimators']),
    max_depth=int(best_params['max_depth']),
    min_samples_split=int(best_params['min_samples_split']),
    random_state=12
)

final_model.fit(x_train, y_train)
print("Model trained successfully on full training set (no subsampling)!")

Model trained successfully on full training set (no subsampling)!


In [8]:
# Test the model on test set
# Calculate c-index
c_index = final_model.score(x_test, y_test)

# Calculate c-index IPCW
tau = y_train["time"].max()
c_index_ipcw = concordance_index_ipcw(y_train, y_test, final_model.predict(x_test), tau=tau)[0]

# Calculate integrated Brier score
surv_funcs = final_model.predict_survival_function(x_test)

# Use robust time filtering
max_time = min(y_train['time'].max(), y_test['time'].max()) * 0.9

# Use percentile-based times (5% to 90%)
times = np.percentile(y_train['time'], np.linspace(5, 90, 10))
# Ensure times are unique and sorted
times = np.unique(times)
# Filter times that are within valid range
times = times[(times > 0) & (times <= max_time)]

preds_at_times = np.array([[fn(t) for t in times] for fn in surv_funcs])
brier_score = integrated_brier_score(y_train, y_test, preds_at_times, times)

print("\nTest Set Performance (No Subsampling):")
print(f"  C-index: {c_index:.4f}")
print(f"  C-index IPCW: {c_index_ipcw:.4f}")
print(f"  Integrated Brier Score: {brier_score:.4f}")


Test Set Performance (No Subsampling):
  C-index: 0.8380
  C-index IPCW: 0.6428
  Integrated Brier Score: 0.0073
